# TP 4 — Inférer, comparer CPU/GPU et préparer CUDA — CORRIGÉ

## Objectif général

Ce corrigé montre comment passer d'un modèle entraîné à la génération de texte, puis comment commencer à raisonner comme un AI Engineer : device, benchmark, opérations coûteuses, futurs kernels CUDA.


## Sources principales du parcours

Ces notebooks se basent principalement sur nanoGPT de Karpathy et sur la documentation officielle PyTorch/Jupyter.

- nanoGPT — README et quick start : https://github.com/karpathy/nanoGPT#quick-start  
  À lire pour comprendre le flux officiel : préparer Shakespeare, entraîner avec `train.py`, puis générer avec `sample.py`.
- nanoGPT — `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py  
  À lire comme colonne vertébrale : `LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`, `GPTConfig`, `GPT.forward`, `GPT.generate`.
- nanoGPT — `train.py` : https://github.com/karpathy/nanoGPT/blob/master/train.py  
  À lire pour la boucle d'entraînement : batch, forward, loss, backward, optimizer, checkpoint.
- nanoGPT — `sample.py` : https://github.com/karpathy/nanoGPT/blob/master/sample.py  
  À lire pour l'inférence : chargement du checkpoint, encodage du prompt, `model.generate`.
- Config Shakespeare caractère : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py  
  À lire pour comprendre les hyperparamètres pédagogiques : `block_size`, `n_layer`, `n_head`, `n_embd`, `dropout`, `max_iters`.
- PyTorch — Tensors : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html  
  À lire pour comprendre pourquoi les données, poids et activations sont des tenseurs.
- PyTorch — `torch.nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html  
  À lire pour comprendre la table qui transforme des indices de tokens en vecteurs.
- PyTorch — `torch.nn.LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html  
  À lire pour comprendre la normalisation utilisée dans chaque bloc.
- PyTorch — `torch.nn.GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html  
  À lire pour comprendre l'activation du MLP.
- PyTorch — `torch.nn.CrossEntropyLoss` : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
  À lire pour comprendre la loss de prédiction du prochain token.
- PyTorch — `torch.nn.functional.scaled_dot_product_attention` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html  
  À lire pour comprendre la version optimisée de l'attention Q/K/V.
- PyTorch — CUDA semantics : https://docs.pytorch.org/docs/stable/notes/cuda.html  
  À lire pour comprendre le placement CPU/GPU des tenseurs et modèles.
- Jupyter — Markdown cells : https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html  
  À lire pour savoir structurer les réponses dans les cellules Markdown.


## Partie 1 — Générer du texte avec `sample.py`

### Question 4.1 — Lancer `sample.py`

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys, os, torch

repo = Path("nanoGPT")
assert repo.exists(), "Clone nanoGPT d'abord."
assert (repo / "out-shakespeare-char-tp/ckpt.pt").exists(), "Exécute le TP 3 pour créer le checkpoint."

cmd = [
    sys.executable, "sample.py",
    "--out_dir=out-shakespeare-char-tp",
    "--device=cpu",
    "--compile=False",
    "--num_samples=2",
    "--max_new_tokens=200",
    "--temperature=0.8",
    "--top_k=50",
]

print("Commande :")
print(" ".join(cmd))
subprocess.run(cmd, cwd=repo, check=True)


### Explication pédagogique

`sample.py` fait l'inférence :
1. charge le checkpoint ;
2. reconstruit le modèle avec les bons `model_args` ;
3. charge les poids ;
4. encode le prompt ;
5. appelle `model.generate` ;
6. décode les tokens générés.

Comme notre entraînement TP est très court, la sortie sera probablement bruitée.  
Ce n'est pas un échec : l'objectif est de comprendre le pipeline.


### Question 4.2 — Modifier le prompt de départ

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path("nanoGPT")
cmd = [
    sys.executable, "sample.py",
    "--out_dir=out-shakespeare-char-tp",
    "--device=cpu",
    "--compile=False",
    "--num_samples=2",
    "--max_new_tokens=200",
    "--temperature=0.8",
    "--top_k=50",
    "--start=ROMEO:",
]

print("Commande :")
print(" ".join(cmd))
subprocess.run(cmd, cwd=repo, check=True)


### Explication pédagogique

Le prompt donne un contexte initial.  
En génération autoregressive, le modèle prédit le prochain token en fonction des tokens déjà présents.

Si tu donnes `ROMEO:`, le modèle commence depuis ces caractères.  
Mais si le modèle est très peu entraîné, il peut ne pas comprendre fortement la structure des personnages.


## Partie 2 — Comprendre `generate`

### Question 4.3 — Lire la boucle de génération

**Solution :**

Dans `GPT.generate`, à chaque itération :

1. `idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]`  
   On garde seulement le contexte maximal autorisé.

2. `logits, _ = self(idx_cond)`  
   On fait un forward pass.

3. `logits = logits[:, -1, :]`  
   On garde seulement les scores du dernier token, car c'est lui qui prédit la suite.

4. `logits = logits / temperature`  
   On ajuste la concentration de la distribution.

5. Si `top_k` est défini, on garde seulement les `k` meilleurs logits.

6. `probs = F.softmax(logits, dim=-1)`  
   On transforme les scores en probabilités.

7. `idx_next = torch.multinomial(probs, num_samples=1)`  
   On échantillonne le prochain token.

8. `idx = torch.cat((idx, idx_next), dim=1)`  
   On ajoute ce token à la séquence et on recommence.

### Explication pédagogique

La génération GPT est une boucle.  
Le modèle ne génère pas 200 tokens en une seule prédiction indépendante.  
Il génère :

```text
token 1 → ajoute au contexte
token 2 → ajoute au contexte
token 3 → ajoute au contexte
...
```

C'est pourquoi l'inférence LLM est souvent coûteuse : chaque nouveau token nécessite un nouveau passage du modèle.


### Question 4.4 — Expérimenter température et top-k

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path("nanoGPT")
settings = [
    ("temp basse", "0.5", "50"),
    ("temp haute", "1.2", "50"),
    ("top_k petit", "0.8", "10"),
    ("top_k grand", "0.8", "100"),
]

for label, temp, top_k in settings:
    print("\n" + "="*80)
    print(label, "| temperature =", temp, "| top_k =", top_k)
    print("="*80)

    cmd = [
        sys.executable, "sample.py",
        "--out_dir=out-shakespeare-char-tp",
        "--device=cpu",
        "--compile=False",
        "--num_samples=1",
        "--max_new_tokens=120",
        f"--temperature={temp}",
        f"--top_k={top_k}",
        "--start=ROMEO:",
    ]
    subprocess.run(cmd, cwd=repo, check=True)


### Explication pédagogique

- Température basse : distribution plus concentrée, sorties plus prévisibles.
- Température haute : distribution plus plate, sorties plus aléatoires.
- `top_k` petit : seuls quelques tokens candidats restent possibles.
- `top_k` grand : davantage de tokens peuvent être tirés.

**Erreur fréquente :**  
Penser que `temperature` améliore l'intelligence du modèle. Non : elle modifie seulement la stratégie d'échantillonnage.


## Partie 3 — Comparer CPU et GPU

### Question 4.5 — Vérifier le device

**Solution :**


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA disponible :", torch.cuda.is_available())
print("Device choisi    :", device)

if torch.cuda.is_available():
    print("Nom du GPU       :", torch.cuda.get_device_name(0))


### Explication pédagogique

Un tenseur PyTorch vit sur un device.  
Une opération entre deux tenseurs exige qu'ils soient sur le même device.

Exemple :
- modèle sur GPU ;
- batch sur CPU ;

=> erreur fréquente.

Pour entraîner ou inférer sur GPU, il faut déplacer le modèle et les données vers `cuda`.


### Question 4.6 — Benchmark simple de matmul

**Solution :**


In [ ]:
import torch, time

def benchmark_matmul(device, n=1024, repeats=20):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)

    # Warmup
    for _ in range(3):
        c = a @ b

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        c = a @ b

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    return (end - start) / repeats

cpu_time = benchmark_matmul("cpu", n=512, repeats=10)
print(f"CPU : {cpu_time*1000:.2f} ms par matmul")

if torch.cuda.is_available():
    gpu_time = benchmark_matmul("cuda", n=512, repeats=10)
    print(f"GPU : {gpu_time*1000:.2f} ms par matmul")
else:
    print("CUDA non disponible : benchmark GPU ignoré.")


### Explication pédagogique

Les gros calculs d'un LLM sont dominés par des multiplications matricielles :
- projections Q/K/V ;
- projection de sortie attention ;
- MLP ;
- projection vers vocabulaire.

Un GPU est conçu pour paralléliser massivement ces opérations.  
Mais il faut faire attention à la mesure : les opérations CUDA sont asynchrones, donc on utilise `torch.cuda.synchronize()` pour chronométrer correctement.

**Erreur fréquente :**  
Mesurer du code GPU sans synchroniser. Le temps affiché peut alors être faux.


## Partie 4 — Préparer le pont vers CUDA

### Question 4.7 — Identifier les opérations candidates CUDA

**Solution priorisée :**

| Priorité | Opération | Pourquoi c'est intéressant |
|---|---|---|
| 1 | Addition élément-wise / résidu | Très simple pour apprendre thread indexing CUDA. |
| 2 | Matmul naïf | Base des projections linéaires et de QKᵀ. |
| 3 | Softmax ligne par ligne | Important pour l'attention ; introduit réduction et stabilité numérique. |
| 4 | Attention simplifiée `softmax(QKᵀ)V` | Relie directement CUDA au cœur de GPT. |
| 5 | LayerNorm | Intéressant pour réduction par ligne et normalisation. |
| 6 | MLP complet | Combine matmul + activation + matmul. |
| 7 | Attention optimisée/fusionnée | Sujet avancé : mémoire, tiling, FlashAttention. |

### Explication pédagogique

Pour apprendre CUDA, il ne faut pas commencer par “réécrire GPT”.  
Il faut isoler les briques :

1. addition ;
2. multiplication élément-wise ;
3. réduction ;
4. matmul ;
5. softmax ;
6. attention.

Ensuite seulement, tu peux relier ces briques aux lignes de `model.py`.

**Lien avec nanoGPT :**
- `q @ k.transpose` → matmul ;
- `F.softmax` → softmax ;
- `att @ v` → matmul ;
- `c_attn`, `c_proj`, MLP → Linear/matmul ;
- `x + ...` → addition résiduelle ;
- `LayerNorm` → réduction + normalisation.

**Phrase à retenir :**

> Pour comprendre le serving LLM, il faut comprendre quelles lignes Python déclenchent en réalité des kernels GPU coûteux.
